# Manifold-Aware Sparse Autoencoders Tutorial

This tutorial demonstrates how to use manifold features in SAELens to represent geometric structures like circular and spherical features.

**Key Concepts:**
- Circular manifolds (S¹): For periodic features like days of the week, months
- Spherical manifolds (S²): For directional features like spatial orientations
- Manifold-aware evaluation metrics: Geodesic preservation, topology preservation
- Manifold-aware SAE architectures: Grouped Latent SAE, Manifold-Parametric SAE

**References:**
- Engels et al. (2025): "Not All Language Model Features Are One-Dimensionally Linear"
- Li et al. (2025): "The Geometry of Concepts: Sparse Autoencoder Feature Structure"
- Michaud et al. (2024): "Understanding SAE Scaling with Feature Manifolds"

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from sae_lens.synthetic.manifolds import (
    ManifoldType,
    ManifoldConfig,
    ManifoldFeatureDictionary,
    generate_circular_manifold,
    generate_spherical_manifold,
    compute_geodesic_distance_circular,
)
from sae_lens.synthetic.manifold_evaluation import (
    detect_manifold_clusters_via_coactivation,
    compute_manifold_alignment_score,
    evaluate_manifold_sae,
)
from sae_lens.saes import GroupedLatentSAE, ManifoldParametricSAE

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## Part 1: Generating Manifold Features

Let's start by generating circular and spherical manifolds and visualizing them.

In [ ]:
# Generate circular manifold (e.g., days of the week)
num_days = 7
days_coords, days_angles = generate_circular_manifold(
    num_points=num_days,
    noise_level=0.05,
    device=device
)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Circular manifold
ax = axes[0]
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
coords_np = days_coords.cpu().numpy()
ax.scatter(coords_np[:, 0], coords_np[:, 1], s=100, alpha=0.6)
for i, day in enumerate(days):
    ax.annotate(day, (coords_np[i, 0], coords_np[i, 1]), 
                xytext=(5, 5), textcoords='offset points')
circle = plt.Circle((0, 0), 1, fill=False, color='gray', linestyle='--')
ax.add_patch(circle)
ax.set_xlim(-1.2, 1.2)
ax.set_ylim(-1.2, 1.2)
ax.set_aspect('equal')
ax.set_title('Circular Manifold: Days of the Week')
ax.grid(True, alpha=0.3)

# Generate spherical manifold
directions_coords, _ = generate_spherical_manifold(
    num_points=50,
    noise_level=0.05,
    device=device
)

# Visualize spherical manifold
ax = axes[1]
dirs_np = directions_coords.cpu().numpy()
ax = fig.add_subplot(122, projection='3d')
ax.scatter(dirs_np[:, 0], dirs_np[:, 1], dirs_np[:, 2], alpha=0.6)
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title('Spherical Manifold: Directions')

plt.tight_layout()
plt.show()

print(f"Circular manifold shape: {days_coords.shape}")
print(f"Spherical manifold shape: {directions_coords.shape}")

## Part 2: Creating a Hybrid Feature Dictionary

Create a feature dictionary with both independent features and manifold features.

In [ ]:
# Configure manifolds
manifold_configs = [
    ManifoldConfig(
        manifold_type=ManifoldType.CIRCULAR,
        num_points=7,  # Days of the week
        embedding_dim=16,
        intrinsic_dim=2,
        noise_level=0.05,
        name="day_of_week"
    ),
    ManifoldConfig(
        manifold_type=ManifoldType.CIRCULAR,
        num_points=12,  # Months of the year
        embedding_dim=16,
        intrinsic_dim=2,
        noise_level=0.05,
        name="month"
    ),
    ManifoldConfig(
        manifold_type=ManifoldType.SPHERICAL,
        num_points=50,  # Spatial directions
        embedding_dim=20,
        intrinsic_dim=3,
        noise_level=0.05,
        name="direction"
    ),
]

# Create hybrid feature dictionary
feature_dict = ManifoldFeatureDictionary(
    num_independent=100,  # 100 standard 1D features
    manifold_configs=manifold_configs,
    hidden_dim=128,  # Activation space dimension
    device=device,
    seed=42
)

print(f"Total features: {feature_dict.num_features}")
print(f"  - Independent: {feature_dict.num_independent}")
print(f"  - Manifolds: {feature_dict.num_manifolds}")
print(f"\nManifold breakdown:")
for meta in feature_dict.get_manifold_metadata():
    print(f"  {meta['name']}: {meta['num_points']} points, "
          f"intrinsic_dim={meta['intrinsic_dim']}, "
          f"indices [{meta['start_idx']}, {meta['end_idx']})")

## Part 3: Generating Synthetic Activations

Generate synthetic activation data with manifold features.

In [ ]:
# Generate sparse feature activations
num_samples = 1000
batch_size = 32

# For each sample, activate a random subset of features
def generate_manifold_activations(num_samples, feature_dict):
    """Generate activations with manifold structure."""
    feature_acts = torch.zeros(num_samples, feature_dict.num_features, device=device)
    
    for i in range(num_samples):
        # Randomly activate ~10 independent features
        num_active_ind = torch.randint(5, 15, (1,)).item()
        active_ind = torch.randperm(feature_dict.num_independent)[:num_active_ind]
        feature_acts[i, active_ind] = torch.rand(num_active_ind, device=device) * 2
        
        # For each manifold, randomly decide if active
        # If active, pick ONE point on the manifold
        for meta in feature_dict.get_manifold_metadata():
            if torch.rand(1).item() < 0.3:  # 30% chance manifold is active
                # Pick a random point on this manifold
                point_idx = torch.randint(0, meta['num_points'], (1,)).item()
                feature_idx = meta['start_idx'] + point_idx
                feature_acts[i, feature_idx] = torch.rand(1, device=device).item() * 3
    
    return feature_acts

# Generate feature activations
feature_acts = generate_manifold_activations(num_samples, feature_dict)

# Generate hidden activations
hidden_acts = feature_dict(feature_acts)

print(f"Feature activations shape: {feature_acts.shape}")
print(f"Hidden activations shape: {hidden_acts.shape}")
print(f"Average L0 (sparsity): {(feature_acts > 0).float().sum(dim=1).mean().item():.2f}")

# Visualize activation statistics
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# L0 distribution
l0_per_sample = (feature_acts > 0).float().sum(dim=1).cpu().numpy()
axes[0].hist(l0_per_sample, bins=30, alpha=0.7, edgecolor='black')
axes[0].set_xlabel('L0 (number of active features)')
axes[0].set_ylabel('Count')
axes[0].set_title('Sparsity Distribution')

# Feature activation frequencies
activation_freq = (feature_acts > 0).float().mean(dim=0).cpu().numpy()
axes[1].plot(activation_freq, alpha=0.7)
axes[1].axvline(100, color='red', linestyle='--', label='Start of manifolds')
axes[1].set_xlabel('Feature index')
axes[1].set_ylabel('Activation frequency')
axes[1].set_title('Feature Activation Frequencies')
axes[1].legend()

# Hidden activation distribution
axes[2].hist(hidden_acts.flatten().cpu().numpy(), bins=50, alpha=0.7, edgecolor='black')
axes[2].set_xlabel('Activation value')
axes[2].set_ylabel('Count')
axes[2].set_title('Hidden Activation Distribution')

plt.tight_layout()
plt.show()

## Part 4: Training a Grouped Latent SAE

Train a Grouped Latent SAE that can learn to represent manifold structures.

In [ ]:
# Create Grouped Latent SAE
sae = GroupedLatentSAE(
    d_in=128,  # Hidden dimension
    d_sae=256,  # Total latents
    num_groups=16,  # 16 groups
    latents_per_group=16,  # 16 latents per group
    lambda_group=1e-3,  # Group sparsity penalty
    lambda_feature=1e-3,  # Feature sparsity penalty
    device=device
)

print(f"SAE architecture: {sae.num_groups} groups × {sae.latents_per_group} latents = {sae.d_sae} total")

# Simple training loop
optimizer = torch.optim.Adam(sae.parameters(), lr=1e-3)
num_epochs = 20
losses = []

for epoch in range(num_epochs):
    epoch_loss = 0.0
    
    for i in range(0, num_samples, batch_size):
        batch = hidden_acts[i:i+batch_size]
        
        optimizer.zero_grad()
        loss_dict = sae.get_loss_dict(batch)
        loss = loss_dict['loss']
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / (num_samples // batch_size)
    losses.append(avg_loss)
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

# Plot training curve
plt.figure(figsize=(8, 5))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('SAE Training Curve')
plt.grid(True, alpha=0.3)
plt.show()

## Part 5: Evaluating Manifold Recovery

Evaluate how well the SAE recovered the manifold structures.

In [ ]:
# Get SAE activations
with torch.no_grad():
    sae_latents = sae.encode(hidden_acts)

# Detect manifold clusters in SAE
clusters = detect_manifold_clusters_via_coactivation(
    sae_latents,
    threshold_correlation=0.2,
    min_cluster_size=2
)

print(f"\nDetected {len(clusters)} potential manifold clusters in SAE:")
for i, cluster in enumerate(clusters[:10]):  # Show first 10
    print(f"  Cluster {i}: {len(cluster)} latents - indices {cluster[:5]}...")

# Compute reconstruction quality
with torch.no_grad():
    reconstructed = sae(hidden_acts)
    mse = ((reconstructed - hidden_acts) ** 2).mean()
    variance_explained = 1 - mse / hidden_acts.var()

print(f"\nReconstruction Metrics:")
print(f"  MSE: {mse.item():.4f}")
print(f"  Variance Explained: {variance_explained.item():.4f}")
print(f"  Average L0: {(sae_latents > 0).float().sum(dim=1).mean().item():.2f}")

## Part 6: Computing Geodesic Distance Preservation

Measure how well the SAE preserves geodesic distances on manifolds.

In [ ]:
# For the circular "day of week" manifold
day_meta = feature_dict.get_manifold_metadata()[0]
day_start = day_meta['start_idx']
day_end = day_meta['end_idx']

# Get samples where day manifold is active
day_active_mask = (feature_acts[:, day_start:day_end] > 0).any(dim=1)
day_samples = hidden_acts[day_active_mask]
day_gt_acts = feature_acts[day_active_mask, day_start:day_end]

print(f"Found {day_samples.shape[0]} samples with active 'day_of_week' manifold")

if day_samples.shape[0] > 10:
    # Get SAE representations
    with torch.no_grad():
        day_sae_latents = sae.encode(day_samples)
    
    # Get ground truth circular coordinates
    days_coords_gt, days_angles_gt = generate_circular_manifold(7, device=device)
    
    # Compute geodesic distances in ground truth
    gt_geodesic = compute_geodesic_distance_circular(days_angles_gt)
    
    # For each detected cluster, compute distance preservation
    if len(clusters) > 0:
        best_cluster = clusters[0]  # Use largest cluster
        
        # Compute distances in SAE space
        cluster_acts = day_sae_latents[:, best_cluster]
        sae_distances = torch.cdist(cluster_acts, cluster_acts)
        
        # Compute correlation
        from scipy.stats import spearmanr
        
        gt_flat = gt_geodesic[torch.triu_indices(7, 7, offset=1)].cpu().numpy()
        
        # Sample pairwise distances
        if sae_distances.shape[0] > 1:
            n = min(sae_distances.shape[0], 50)  # Sample up to 50
            indices = torch.randperm(sae_distances.shape[0])[:n]
            sampled_sae = sae_distances[indices][:, indices]
            sae_flat = sampled_sae[torch.triu_indices(n, n, offset=1)].cpu().numpy()
            
            if len(sae_flat) > 2 and len(gt_flat) > 2:
                # Repeat gt_flat to match size (crude approximation)
                gt_expanded = np.tile(gt_flat, (len(sae_flat) // len(gt_flat)) + 1)[:len(sae_flat)]
                corr, pval = spearmanr(gt_expanded, sae_flat)
                
                print(f"\nGeodesic Distance Preservation:")
                print(f"  Spearman correlation: {corr:.3f} (p={pval:.3e})")
                print(f"  Interpretation: {'Good' if corr > 0.5 else 'Poor'} preservation")

## Part 7: Visualizing SAE Decoder Geometry

Analyze the geometric structure of the SAE decoder weights.

In [ ]:
from sae_lens.synthetic.manifold_evaluation import analyze_decoder_subspace_geometry

# Analyze geometry of detected clusters
decoder_weights = sae.W_dec.data.T  # Shape: (d_hidden, num_latents)

print("\nDecoder Geometry Analysis:")
for i, cluster in enumerate(clusters[:5]):
    result = analyze_decoder_subspace_geometry(decoder_weights, cluster)
    
    print(f"\nCluster {i} ({len(cluster)} latents):")
    print(f"  Intrinsic dimensionality: {result['intrinsic_dim']}")
    print(f"  Top singular values: {result['singular_values'][:3]}")
    
    if result['intrinsic_dim'] == 2:
        print(f"  Circularity score: {result['circularity_score']:.3f}")
        interpretation = "circular" if result['circularity_score'] > 0.7 else "non-circular"
        print(f"  Interpretation: Looks {interpretation}")
    
    if result['intrinsic_dim'] == 3:
        print(f"  Sphericity score: {result['sphericity_score']:.3f}")
        interpretation = "spherical" if result['sphericity_score'] > 0.7 else "non-spherical"
        print(f"  Interpretation: Looks {interpretation}")

## Summary

In this tutorial, we:

1. ✅ Generated circular and spherical manifolds
2. ✅ Created a hybrid feature dictionary with independent + manifold features
3. ✅ Generated synthetic activation data with manifold structure
4. ✅ Trained a Grouped Latent SAE
5. ✅ Detected manifold clusters in the learned SAE
6. ✅ Evaluated geodesic distance preservation
7. ✅ Analyzed decoder geometry

**Key Takeaways:**
- Manifold features are multi-dimensional (circular = 2D, spherical = 3D)
- Standard SAEs may "tile" manifolds with many latents instead of learning the geometry
- Manifold-aware architectures (GroupedLatentSAE, ManifoldParametricSAE) can learn more efficient representations
- Evaluation requires geometric metrics (geodesic preservation, topology) beyond standard MCC/F1

**Next Steps:**
- Try the Manifold-Parametric SAE for explicit geometric parameterization
- Experiment with different numbers of groups and latents
- Test on real LLM activations to find calendar features (days, months)
- Compare manifold-aware vs. standard SAE performance